In [58]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import mlflow
import random
import glob
import json
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from datetime import datetime

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

random_seed = 42    
torch.manual_seed(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)


# Make sure mlflow ui is running: run 'mlflow ui' in terminal first
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("CV_remove_videos")
mlflow.enable_system_metrics_logging()

Using device: mps


## Data Loading

In [27]:
def input_target_split(dataframe):
    input_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y") or c.endswith("_z"):
            input_cols.append(c)

    input_data  = dataframe[input_cols]
    target_data = dataframe[["running_video"]]  # Binary label: 1 = squat frame, 0 = not
    return input_data, target_data


def extract_sequences(file_list, seq_length=30, stride=10):
    """
    Extract sliding-window sequences from a list of CSV video files.

    Args:
        file_list  : list of CSV file paths
        seq_length : number of frames per sequence
        stride     : step size between windows (< seq_length = overlap)

    Returns:
        X : (n_sequences, seq_length, 39)
        y : (n_sequences, seq_length, 1)
    """
    all_X, all_y = [], []
    for file_path in file_list:
        df   = pd.read_csv(file_path)
        X_df, y_df = input_target_split(df)
        X_np = X_df.values.astype(np.float32)
        y_np = y_df.values.astype(np.float32)  # (n_frames, 1)
        n_frames = len(df)
        for i in range(0, n_frames - seq_length + 1, stride):
            all_X.append(torch.tensor(X_np[i:i + seq_length]))
            all_y.append(torch.tensor(y_np[i:i + seq_length]))
    if not all_X:
        return torch.empty(0, seq_length, 39), torch.empty(0, seq_length, 1)
    return torch.stack(all_X), torch.stack(all_y)


def mirror_sequences_tensor(X, Y):
    """
    Mirror sequences where X contains (x,y,z) per joint.
    
    X: (n_seq, seq_len, 39)  # 13 joints * 3 coords
    Y: labels (unchanged)
    """

    mirror_pairs = [
        (1, 3),
        (2, 4),
        (5, 6),
        (7, 8),
        (9, 10),
        (11, 12),
    ]

    X_mirrored = X.clone()

    for left, right in mirror_pairs:
        # indices for (x,y,z)
        lx, ly, lz = 3*left, 3*left+1, 3*left+2
        rx, ry, rz = 3*right, 3*right+1, 3*right+2

        # --- X (mirror: negate + swap)
        tmp = X_mirrored[:, :, lx].clone()
        X_mirrored[:, :, lx] = -X_mirrored[:, :, rx]
        X_mirrored[:, :, rx] = -tmp

        # --- Y (swap only)
        tmp = X_mirrored[:, :, ly].clone()
        X_mirrored[:, :, ly] = X_mirrored[:, :, ry]
        X_mirrored[:, :, ry] = tmp

        # --- Z (swap only)
        tmp = X_mirrored[:, :, lz].clone()
        X_mirrored[:, :, lz] = X_mirrored[:, :, rz]
        X_mirrored[:, :, rz] = tmp

    return X_mirrored, Y

## Model

In [33]:
class Recurrent_classifier(nn.Module):
    def __init__(self, hidden_layers: list, layer_type="LSTM", dropout=0):
        super().__init__()

        input_size = 39
        rnn_class  = nn.LSTM if layer_type == "LSTM" else nn.GRU

        self.rnns  = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.norms = nn.ModuleList()  # Add normalization layers

        sizes = [input_size] + hidden_layers
        for i in range(len(hidden_layers)):
            in_size = sizes[i] * 2 if i > 0 else sizes[i]  # *2 because bidirectional doubles output
            self.rnns.append(rnn_class(in_size, sizes[i + 1], batch_first=True, bidirectional=True))
            self.drops.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            
            # Add LayerNorm after each RNN (normalize over the hidden dimension)
            hidden_dim = sizes[i + 1] * 2  # *2 for bidirectional
            self.norms.append(nn.LayerNorm(hidden_dim))

        self.fc_out = nn.Linear(hidden_layers[-1] * 2, 1)  # *2 for bidirectional

    def forward(self, x):
        for rnn, drop, norm in zip(self.rnns, self.drops, self.norms):
            x, _ = rnn(x)
            x = norm(x)  # Apply layer normalization
            x = drop(x)
    
        return self.fc_out(x)  # (batch, seq_len, 1) — raw logits

## Cross-Validation Training

In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

def run_cv(
    datafolder,
    k             = 10,
    seq_length    = 30,
    stride        = 10,
    hidden_layers = [64, 32],
    dropout       = 0.2,
    epochs        = 30,
    lr            = 0.001,
    batch_size    = 32,
    name_of_run   = "",
):
    
    csv_files = np.array(glob.glob(os.path.join(datafolder, "*.csv")))
    random.seed(random_seed)
    random.shuffle(csv_files)
    print(f"Found {len(csv_files)} video files — running {k}-fold CV")

    kf           = KFold(n_splits=k, shuffle=True, random_state=random_seed)
    fold_results = []

    for fold, (train_val_idx, test_idx) in enumerate(kf.split(csv_files)):
        print(f"\n{'='*50}")
        print(f"  FOLD {fold + 1} / {k}")
        print(f"{'='*50}")

        train_val_files = csv_files[train_val_idx]
        test_files      = csv_files[test_idx]

        n_val       = max(1, int(len(train_val_files) * 0.15))
        val_files   = train_val_files[:n_val]
        train_files = train_val_files[n_val:]

        print(f"  Train videos: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

        # ── Raw sequences (unscaled) ───────────────────────────
        train_x, train_y = extract_sequences(train_files, seq_length, stride)
        val_x,   val_y   = extract_sequences(val_files,   seq_length, stride)
        test_x,  test_y  = extract_sequences(test_files,  seq_length, stride)

        train_x, train_y = mirror_sequences_tensor(train_x, train_y)

        X_mirror, Y_mirror = mirror_sequences_tensor(train_x, train_y)

        train_x = torch.cat([train_x, X_mirror], dim=0)
        train_y = torch.cat([train_y, Y_mirror], dim=0)


        

        print(f"  Sequences — Train: {len(train_x)} | Val: {len(val_x)} | Test: {len(test_x)}")

        # ── StandardScaler — fit on train only ─────────────────
        # Reshape to 2D (n_samples * seq_length, n_features) to fit scaler
        scaler = StandardScaler()

        train_shape = train_x.shape  # (n, seq_len, 39)
        val_shape   = val_x.shape
        test_shape  = test_x.shape

        train_x = torch.tensor(
            scaler.fit_transform(train_x.numpy().reshape(-1, train_shape[-1]))
        ).reshape(train_shape).float()

        # transform only — never fit on val/test
        val_x = torch.tensor(
            scaler.transform(val_x.numpy().reshape(-1, val_shape[-1]))
        ).reshape(val_shape).float()

        test_x = torch.tensor(
            scaler.transform(test_x.numpy().reshape(-1, test_shape[-1]))
        ).reshape(test_shape).float()

        # ── Dataloaders ────────────────────────────────────────
        train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(TensorDataset(val_x,   val_y),   batch_size=batch_size)
        test_loader  = DataLoader(TensorDataset(test_x,  test_y),  batch_size=batch_size)

        # ── Model ──────────────────────────────────────────────
        model     = Recurrent_classifier(hidden_layers, dropout=dropout).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.BCEWithLogitsLoss()

        # ── MLflow run ─────────────────────────────────────────
        with mlflow.start_run(run_name=f"{name_of_run} fold_{fold + 1}"):

            mlflow.log_params({
                "fold":          fold + 1,
                "k":             k,
                "hidden_layers": str(hidden_layers),
                "dropout":       dropout,
                "seq_length":    seq_length,
                "stride":        stride,
                "epochs":        epochs,
                "lr":            lr,
                "batch_size":    batch_size,
                "train_videos":  len(train_files),
                "val_videos":    len(val_files),
                "test_videos":   len(test_files),
                "train_seqs":    len(train_x),
                "val_seqs":      len(val_x),
                "test_seqs":     len(test_x),
                "scaler":        "StandardScaler",
            })

            best_val_loss   = float("inf")
            best_model_path = f"best_fold_{fold + 1}.pt"

            for epoch in range(epochs):

                # ── Train epoch ────────────────────────────────
                model.train()
                train_loss, train_correct, train_total = 0.0, 0, 0
                train_preds_all, train_labels_all      = [], []

                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = criterion(pred, yb)
                    loss.backward()
                    optimizer.step()

                    train_loss    += loss.item() * xb.size(0)
                    preds_binary   = (torch.sigmoid(pred) > 0.5)
                    train_correct += (preds_binary == yb.bool()).sum().item()
                    train_total   += yb.numel()

                    train_preds_all.append(preds_binary.cpu().flatten())
                    train_labels_all.append(yb.bool().cpu().flatten())

                # ── Val epoch ──────────────────────────────────
                model.eval()
                val_loss, val_correct, val_total  = 0.0, 0, 0
                val_preds_all, val_labels_all      = [], []

                with torch.no_grad():
                    for xb, yb in val_loader:
                        xb, yb = xb.to(device), yb.to(device)
                        pred   = model(xb)
                        loss   = criterion(pred, yb)

                        val_loss    += loss.item() * xb.size(0)
                        preds_binary = (torch.sigmoid(pred) > 0.5)
                        val_correct += (preds_binary == yb.bool()).sum().item()
                        val_total   += yb.numel()

                        val_preds_all.append(preds_binary.cpu().flatten())
                        val_labels_all.append(yb.bool().cpu().flatten())

                avg_train_loss = train_loss / train_total
                avg_val_loss   = val_loss   / val_total
                train_acc      = train_correct / train_total
                val_acc        = val_correct   / val_total

                # F1 per epoch
                train_f1 = f1_score(
                    torch.cat(train_labels_all).numpy(),
                    torch.cat(train_preds_all).numpy(),
                    zero_division=0
                )
                val_f1 = f1_score(
                    torch.cat(val_labels_all).numpy(),
                    torch.cat(val_preds_all).numpy(),
                    zero_division=0
                )

                # ── Log per epoch ──────────────────────────────
                mlflow.log_metrics({
                    "train_loss": avg_train_loss,
                    "val_loss":   avg_val_loss,
                    "train_acc":  train_acc,
                    "val_acc":    val_acc,
                    "train_f1":   train_f1,
                    "val_f1":     val_f1,
                }, step=epoch)

                if (epoch + 1) % 10 == 0:
                    print(
                        f"  Epoch {epoch + 1:3d}/{epochs} "
                        f"| train_loss: {avg_train_loss:.4f} "
                        f"| val_loss: {avg_val_loss:.4f} "
                        f"| train_acc: {train_acc:.4f} "
                        f"| val_acc: {val_acc:.4f} "
                        f"| train_f1: {train_f1:.4f} "
                        f"| val_f1: {val_f1:.4f}"
                    )

                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    torch.save(model.state_dict(), best_model_path)

            # ── Test with best checkpoint ───────────────────────
            model.load_state_dict(torch.load(best_model_path))
            model.eval()

            test_loss, test_correct, test_total  = 0.0, 0, 0
            test_preds_all, test_labels_all       = [], []

            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    pred   = model(xb)
                    loss   = criterion(pred, yb)

                    test_loss    += loss.item() * xb.size(0)
                    preds_binary  = (torch.sigmoid(pred) > 0.5)
                    test_correct += (preds_binary == yb.bool()).sum().item()
                    test_total   += yb.numel()

                    test_preds_all.append(preds_binary.cpu().flatten())
                    test_labels_all.append(yb.bool().cpu().flatten())

            test_acc  = test_correct / test_total
            test_loss = test_loss    / test_total
            test_f1   = f1_score(
                torch.cat(test_labels_all).numpy(),
                torch.cat(test_preds_all).numpy(),
                zero_division=0
            )

            mlflow.log_metrics({
                "test_acc":      test_acc,
                "test_loss":     test_loss,
                "test_f1":       test_f1,
                "best_val_loss": best_val_loss,
            })

            mlflow.log_artifact(best_model_path)

            print(f"\n  Fold {fold + 1} test_acc: {test_acc:.4f} | test_loss: {test_loss:.4f} | test_f1: {test_f1:.4f}")
            fold_results.append({"acc": test_acc, "f1": test_f1})

    # ── Summary ────────────────────────────────────────────────
    accs = [r["acc"] for r in fold_results]
    f1s  = [r["f1"]  for r in fold_results]

    print(f"\n{'='*50}")
    print(f"CV RESULTS ({k} folds)")
    for i, r in enumerate(fold_results):
        print(f"  Fold {i + 1}: acc={r['acc']:.4f} | f1={r['f1']:.4f}")
    print(f"  Acc  — Mean: {np.mean(accs):.4f} | Std: {np.std(accs):.4f}")
    print(f"  F1   — Mean: {np.mean(f1s):.4f}  | Std: {np.std(f1s):.4f}")
    print(f"{'='*50}")

    return fold_results

## Run

In [37]:
DATAFOLDER = "../../MainProject/Data/mediapipe_removed_non_squat"  

fold_results = run_cv(
    datafolder    = DATAFOLDER,
    k             = 10,
    seq_length    = 30,
    stride        = 10,
    hidden_layers = [64,32],
    dropout       = 0.2,
    epochs        = 35,
    lr            = 0.0008,
    batch_size    = 32,
    name_of_run   = "clean_baseline_v2"
)

Found 175 video files — running 10-fold CV

  FOLD 1 / 10
  Train videos: 134 | Val: 23 | Test: 18


2026/05/04 16:23:25 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/05/04 16:23:25 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


  Sequences — Train: 5182 | Val: 420 | Test: 294
  Epoch  10/35 | train_loss: 0.0050 | val_loss: 0.0056 | train_acc: 0.9358 | val_acc: 0.9278 | train_f1: 0.9537 | val_f1: 0.9471
  Epoch  20/35 | train_loss: 0.0039 | val_loss: 0.0063 | train_acc: 0.9501 | val_acc: 0.9302 | train_f1: 0.9636 | val_f1: 0.9490
  Epoch  30/35 | train_loss: 0.0032 | val_loss: 0.0069 | train_acc: 0.9599 | val_acc: 0.9169 | train_f1: 0.9707 | val_f1: 0.9381


2026/05/04 16:24:13 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/05/04 16:24:13 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



  Fold 1 test_acc: 0.9103 | test_loss: 0.0081 | test_f1: 0.9441
🏃 View run clean_baseline_v2 fold_1 at: http://127.0.0.1:5000/#/experiments/6/runs/fd8a1a3a39964fef9d7a0ee13afee7db
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6

  FOLD 2 / 10
  Train videos: 134 | Val: 23 | Test: 18


...


  Fold 9 test_acc: 0.9233 | test_loss: 0.0066 | test_f1: 0.9447
🏃 View run clean_baseline_v2 fold_9 at: http://127.0.0.1:5000/#/experiments/6/runs/8b4ff71157794eba8ae0b2703aed73ea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6

  FOLD 10 / 10
  Train videos: 135 | Val: 23 | Test: 17


2026/05/04 16:30:56 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/05/04 16:30:56 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


  Sequences — Train: 5092 | Val: 415 | Test: 344
  Epoch  10/35 | train_loss: 0.0051 | val_loss: 0.0058 | train_acc: 0.9318 | val_acc: 0.9351 | train_f1: 0.9511 | val_f1: 0.9544
  Epoch  20/35 | train_loss: 0.0039 | val_loss: 0.0053 | train_acc: 0.9494 | val_acc: 0.9367 | train_f1: 0.9633 | val_f1: 0.9550
  Epoch  30/35 | train_loss: 0.0036 | val_loss: 0.0057 | train_acc: 0.9543 | val_acc: 0.9328 | train_f1: 0.9668 | val_f1: 0.9520


2026/05/04 16:31:45 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/05/04 16:31:45 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



  Fold 10 test_acc: 0.9222 | test_loss: 0.0082 | test_f1: 0.9440
🏃 View run clean_baseline_v2 fold_10 at: http://127.0.0.1:5000/#/experiments/6/runs/f0312e5abe114cfebf13c24fe870c912
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6

CV RESULTS (10 folds)
  Fold 1: acc=0.9103 | f1=0.9441
  Fold 2: acc=0.9121 | f1=0.9429
  Fold 3: acc=0.9226 | f1=0.9505
  Fold 4: acc=0.9229 | f1=0.9407
  Fold 5: acc=0.9148 | f1=0.9358
  Fold 6: acc=0.9268 | f1=0.9382
  Fold 7: acc=0.9195 | f1=0.9454
  Fold 8: acc=0.9246 | f1=0.9427
  Fold 9: acc=0.9233 | f1=0.9447
  Fold 10: acc=0.9222 | f1=0.9440
  Acc  — Mean: 0.9199 | Std: 0.0053
  F1   — Mean: 0.9429  | Std: 0.0038


In [51]:
def save_candidate_model(model, model_name, candidates_dir):
    os.makedirs(candidates_dir, exist_ok=True)
    path = os.path.join(candidates_dir, f"{model_name}.pt")
    torch.save(model.state_dict(), path)
    print(f"  📁 Saved candidate model: {path}")
    return path

def load_champion_info(metadata_dir):
    path = os.path.join(metadata_dir, "champion_info.json")
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None

def save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters):
    os.makedirs(champion_dir, exist_ok=True)
    os.makedirs(metadata_dir, exist_ok=True)
    
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    torch.save(model.state_dict(), model_path)

    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "f1": float(f1),
        "recall": float(recall),
        "precision": float(precision),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print(f"  🏆 New champion model saved! (F1: {f1:.4f})")

def update_champion(metadata_dir, champion_dir, model, model_name, f1, recall, precision, hyperparameters):
    current = load_champion_info(metadata_dir)

    if current is None:
        print("  No champion found → saving first model")
        save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters)
        return True
    elif f1 > current["f1"]:
        print(f"  ✅ New model is better! (F1: {f1:.4f} > {current['f1']:.4f})")
        save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters)
        return True
    else:
        print(f"  ❌ Model NOT better (F1: {f1:.4f} < {current['f1']:.4f})")
        return False

def update_champion_with_test_results(metadata_dir, test_metrics):
    """Add test results to champion info after evaluation"""
    info_path = os.path.join(metadata_dir, "champion_info.json")
    
    if os.path.exists(info_path):
        with open(info_path, "r") as f:
            info = json.load(f)
        
        info["test_results"] = {
            "f1": float(test_metrics["f1"]),
            "accuracy": float(test_metrics["accuracy"]),
            "precision": float(test_metrics["precision"]),
            "recall": float(test_metrics["recall"]),
            "auc": float(test_metrics.get("auc", 0)),
            "evaluated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        with open(info_path, "w") as f:
            json.dump(info, f, indent=2)
        print("  ✅ Champion info updated with test results")
    else:
        print("  ⚠️ No champion info found to update")

In [59]:
def train_final_model(
    datafolder,
    seq_length    = 30,
    stride        = 10,
    hidden_layers = [64, 32],
    dropout       = 0.1,
    epochs        = 40,
    lr            = 0.001,
    batch_size    = 32,
    test_size     = 0.15,
    run_name      = "final_model",
    random_seed   = 42
):
    

    

    
    # ── Load all CSV files ────────────────────────────────────
    csv_files = np.array(glob.glob(os.path.join(datafolder, "*.csv")))
    random.seed(random_seed)
    random.shuffle(csv_files)
    print(f"Found {len(csv_files)} video files")
    
    # ── Split into train + test ───────────────────────────────
    n_test = max(1, int(len(csv_files) * test_size))
    train_files = csv_files[n_test:]
    test_files = csv_files[:n_test]
    
    print(f"Train videos: {len(train_files)}")
    print(f"Test videos: {len(test_files)}")
    

    train_x, train_y = extract_sequences(train_files, seq_length, stride)

    train_x_mirror, train_y_mirror = mirror_sequences_tensor(train_x, train_y)
    
    train_x = torch.cat([train_x, train_x_mirror], dim=0)
    train_y = torch.cat([train_y, train_y_mirror], dim=0)
    
    print("Extracting test sequences...")
    test_x, test_y = extract_sequences(test_files, seq_length, stride)
    print(f"Test sequences: {len(test_x)}")
    
    # ── Normalize data ────────────────────────────────────────
    print("Normalizing data...")
    scaler = StandardScaler()
    
    train_shape = train_x.shape
    test_shape = test_x.shape
    
    train_x_flat = train_x.numpy().reshape(-1, train_shape[-1])
    train_x_scaled = scaler.fit_transform(train_x_flat)
    train_x = torch.tensor(train_x_scaled).reshape(train_shape).float()
    
    test_x_flat = test_x.numpy().reshape(-1, test_shape[-1])
    test_x_scaled = scaler.transform(test_x_flat)
    test_x = torch.tensor(test_x_scaled).reshape(test_shape).float()
    
    # ── Create data loaders ───────────────────────────────────
    train_loader = DataLoader(
        TensorDataset(train_x, train_y),
        batch_size=batch_size,
        shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(test_x, test_y),
        batch_size=batch_size
    )
    
    # ── Model ─────────────────────────────────────────────────
    model = Recurrent_classifier(hidden_layers, dropout=dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    
    # ── MLflow tracking ───────────────────────────────────────
    hyperparameters = {
        "hidden_layers": hidden_layers,
        "dropout": dropout,
        "seq_length": seq_length,
        "stride": stride,
        "epochs": epochs,
        "lr": lr,
        "batch_size": batch_size,
        "train_videos": len(train_files),
        "test_videos": len(test_files),
        "train_sequences": len(train_x),
        "test_sequences": len(test_x),
        "mirror_augmentation": True,
    }
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(hyperparameters)
        
        best_test_f1 = 0.0
        best_test_acc = 0.0
        best_test_precision = 0.0
        best_test_recall = 0.0
        best_epoch = 0
        best_model_state = None
        
        # ── Training loop ─────────────────────────────────────
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0
            train_preds = []
            train_labels = []
            
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred, yb)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item() * xb.size(0)
                preds_binary = (torch.sigmoid(pred) > 0.5)
                train_correct += (preds_binary == yb.bool()).sum().item()
                train_total += yb.numel()
                
                train_preds.extend(preds_binary.cpu().flatten().numpy())
                train_labels.extend(yb.bool().cpu().flatten().numpy())
            
            avg_train_loss = train_loss / train_total
            train_acc = train_correct / train_total
            train_f1 = f1_score(train_labels, train_preds, zero_division=0)
            
            # ── Evaluate on test set ──────────────────────────
            model.eval()
            test_loss = 0.0
            test_correct = 0
            test_total = 0
            test_preds = []
            test_labels = []
            test_probs = []
            
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    pred = model(xb)
                    loss = criterion(pred, yb)
                    
                    test_loss += loss.item() * xb.size(0)
                    probs = torch.sigmoid(pred)
                    preds_binary = (probs > 0.5)
                    test_correct += (preds_binary == yb.bool()).sum().item()
                    test_total += yb.numel()
                    
                    test_preds.extend(preds_binary.cpu().flatten().numpy())
                    test_labels.extend(yb.bool().cpu().flatten().numpy())
                    test_probs.extend(probs.cpu().flatten().numpy())
            
            avg_test_loss = test_loss / test_total
            test_acc = test_correct / test_total
            test_f1 = f1_score(test_labels, test_preds, zero_division=0)
            test_precision = precision_score(test_labels, test_preds, zero_division=0)
            test_recall = recall_score(test_labels, test_preds, zero_division=0)
            test_auc = roc_auc_score(test_labels, test_probs)
            
            # Log metrics
            mlflow.log_metrics({
                "train_loss": avg_train_loss,
                "train_acc": train_acc,
                "train_f1": train_f1,
                "test_loss": avg_test_loss,
                "test_acc": test_acc,
                "test_f1": test_f1,
                "test_precision": test_precision,
                "test_recall": test_recall,
                "test_auc": test_auc,
            }, step=epoch)
            
            # Track best model
            if test_f1 > best_test_f1:
                best_test_f1 = test_f1
                best_test_acc = test_acc
                best_test_precision = test_precision
                best_test_recall = test_recall
                best_epoch = epoch
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            
            # Print progress
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  Epoch {epoch + 1:3d}/{epochs} | "
                      f"train_loss: {avg_train_loss:.4f} | "
                      f"train_f1: {train_f1:.4f} | "
                      f"test_f1: {test_f1:.4f} | "
                      f"test_acc: {test_acc:.4f}")
        
        # ── Restore best model ─────────────────────────────────
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            print(f"\n✅ Restored best model from epoch {best_epoch}")
            print(f"   Best test F1: {best_test_f1:.4f}")
            print(f"   Best test Acc: {best_test_acc:.4f}")
        
        # ── Save candidate model ───────────────────────────────
        candidates_dir = "candidates"
        candidate_name = f"{run_name}_test_f1_{best_test_f1:.4f}"
        save_candidate_model(model, candidate_name, candidates_dir)
        
        # ── Update champion model ──────────────────────────────
        metadata_dir = "champion_metadata"
        champion_dir = "champion"
        
        was_champion = update_champion(
            metadata_dir=metadata_dir,
            champion_dir=champion_dir,
            model=model,
            model_name=run_name,
            f1=best_test_f1,  # Using test F1 for champion selection
            recall=best_test_recall,
            precision=best_test_precision,
            hyperparameters=hyperparameters
        )
        
        # ── Update champion with full test metrics ─────────────
        test_metrics = {
            "f1": best_test_f1,
            "accuracy": best_test_acc,
            "precision": best_test_precision,
            "recall": best_test_recall,
            "auc": test_auc
        }
        update_champion_with_test_results(metadata_dir, test_metrics)
        
        # ── Save final model with scaler ───────────────────────
        final_model_path = f"{run_name}_final.pt"
        torch.save({
            'model_state_dict': model.state_dict(),
            'model_config': {
                'hidden_layers': hidden_layers,
                'dropout': dropout,
                'layer_type': 'LSTM',
                'input_size': 39,
            },
            'scaler': scaler,
            'hyperparameters': hyperparameters,
            'test_metrics': test_metrics,
        }, final_model_path)
        print(f"  💾 Saved final model to {final_model_path}")
        
        # Log final model to MLflow
        mlflow.log_artifact(final_model_path)
        
        print(f"\n{'='*60}")
        print(f"✅ FINAL RESULTS for {run_name}")
        print(f"  Test Accuracy:  {best_test_acc:.4f}")
        print(f"  Test F1:        {best_test_f1:.4f}")
        print(f"  Test Precision: {best_test_precision:.4f}")
        print(f"  Test Recall:    {best_test_recall:.4f}")
        print(f"  Test AUC:       {test_auc:.4f}")
        if was_champion:
            print("  🏆 THIS IS THE NEW CHAMPION!")
        print(f"{'='*60}")
        
    return model, scaler, test_metrics

In [60]:
# ── Run the final model ───────────────────────────────────────
if __name__ == "__main__":
    DATAFOLDER = "../../MainProject/Data/mediapipe_removed_non_squat"

    
    model, scaler, test_acc, test_f1 = train_final_model(
        datafolder    = DATAFOLDER,
        seq_length    = 30,
        stride        = 10,
        hidden_layers = [64,32],
        dropout       = 0.2,
        epochs        = 35,
        lr            = 0.0008,
        batch_size    = 32,
)
        
    
    
    print(f"\n✅ Final model trained!")
    print(f"   Test Accuracy: {test_acc:.4f}")
    print(f"   Test F1 Score: {test_f1:.4f}")

Found 175 video files
Train videos: 149
Test videos: 26


2026/05/04 17:01:00 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/05/04 17:01:00 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Extracting test sequences...
Test sequences: 470
Normalizing data...
  Epoch   1/35 | train_loss: 0.0095 | train_f1: 0.9107 | test_f1: 0.9343 | test_acc: 0.9044
  Epoch   5/35 | train_loss: 0.0059 | train_f1: 0.9450 | test_f1: 0.9480 | test_acc: 0.9265
  Epoch  10/35 | train_loss: 0.0050 | train_f1: 0.9531 | test_f1: 0.9414 | test_acc: 0.9189
  Epoch  15/35 | train_loss: 0.0046 | train_f1: 0.9578 | test_f1: 0.9449 | test_acc: 0.9231
  Epoch  20/35 | train_loss: 0.0040 | train_f1: 0.9628 | test_f1: 0.9482 | test_acc: 0.9274
  Epoch  25/35 | train_loss: 0.0038 | train_f1: 0.9650 | test_f1: 0.9514 | test_acc: 0.9321
  Epoch  30/35 | train_loss: 0.0032 | train_f1: 0.9713 | test_f1: 0.9400 | test_acc: 0.9177


2026/05/04 17:01:55 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/05/04 17:01:55 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


  Epoch  35/35 | train_loss: 0.0031 | train_f1: 0.9712 | test_f1: 0.9393 | test_acc: 0.9170

✅ Restored best model from epoch 26
   Best test F1: 0.9558
   Best test Acc: 0.9383
  📁 Saved candidate model: candidates/final_model_test_f1_0.9558.pt
  ✅ New model is better! (F1: 0.9558 > 0.9375)
  🏆 New champion model saved! (F1: 0.9558)
  ✅ Champion info updated with test results
  💾 Saved final model to final_model_final.pt

✅ FINAL RESULTS for final_model
  Test Accuracy:  0.9383
  Test F1:        0.9558
  Test Precision: 0.9334
  Test Recall:    0.9793
  Test AUC:       0.9742
  🏆 THIS IS THE NEW CHAMPION!
🏃 View run final_model at: http://127.0.0.1:5000/#/experiments/6/runs/97e9314f13b14c0888a523a6ad325716
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


ValueError: not enough values to unpack (expected 4, got 3)